# OmniBench shards -> one tar per owner (for the GPU server)

Reads both shard folders with the **Drive API** (32 parallel requests; no Drive mount, no shortcut needed), checks every
episode against its `_COMPLETE.json`, writes one uncompressed tar per owner (`<owner>/episodes/<id>/...`, the layout
`ppl/` expects under `data/raw/`), uploads it to `My Drive/OneData_transfer/`, and prints **link-shared** download
links (+ the `omni_v1` cache) for `gdown` on the server. Re-running resumes: finished downloads and uploads are skipped.

Runtime: CPU is enough. Uses ~13 GB of local disk.

In [1]:
# ---- configuration
FOLDERS = {"Lee": "1pydCIgnGHB5sx34bdXrAsTJScBljDuPp",          # folder IDs from the Drive links
           "Lazizbek": "1rv2jkjhtWdtFrPEvNojBvsH8Ht1NCnLu"}
OWNER_TASKS = {"Lee": ("T1", "T3", "T5", "T8"), "Lazizbek": ("T2", "T4", "T6", "T7")}
OUT_FOLDER_NAME = "OneData_transfer"      # created in My Drive
CACHE_NAME = "omni_v1_cache.pkl"          # also shared, if found in My Drive
SHARE_PUBLIC = True                       # "anyone with the link can view" on the uploaded files (needed by gdown)
WORKERS = 32
STAGE, OUT = "/content/stage", "/content/out"

In [2]:
import os, io, json, time, threading, subprocess
from concurrent.futures import ThreadPoolExecutor
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from tqdm.auto import tqdm

CREDS, _ = google.auth.default()
_local = threading.local()
def svc():                                # one client per thread (httplib2 is not thread-safe)
    if not hasattr(_local, "s"): _local.s = build("drive", "v3", credentials=CREDS, cache_discovery=False)
    return _local.s
FOLDER, SHORTCUT = "application/vnd.google-apps.folder", "application/vnd.google-apps.shortcut"
ALL = dict(supportsAllDrives=True)

def children(fid):
    out, tok = [], None
    while True:
        r = svc().files().list(q=f"'{fid}' in parents and trashed=false", pageSize=1000, pageToken=tok,
                               fields="nextPageToken, files(id,name,mimeType,size,shortcutDetails)",
                               includeItemsFromAllDrives=True, **ALL).execute(num_retries=5)
        for f in r.get("files", []):
            if f["mimeType"] == SHORTCUT:
                t = f["shortcutDetails"]; f = dict(f, id=t["targetId"], mimeType=t["targetMimeType"], size=None)
            out.append(f)
        tok = r.get("nextPageToken")
        if not tok: return out

def walk(root_id):
    """Parallel breadth-first listing -> [(path, id, size)]; shortcuts are followed."""
    files, level, seen = [], [(root_id, "")], {root_id}
    with ThreadPoolExecutor(WORKERS) as pool:
        while level:
            nxt = []
            for (fid, rel), kids in zip(level, pool.map(lambda x: children(x[0]), level)):
                for k in kids:
                    p = rel + k["name"]
                    if k["mimeType"] == FOLDER:
                        if k["id"] not in seen: seen.add(k["id"]); nxt.append((k["id"], p + "/"))
                    elif not k["mimeType"].startswith("application/vnd.google-apps"):
                        files.append((p, k["id"], int(k["size"]) if k.get("size") else None))
            level = nxt
    return files

def select(files, owner):
    """Keep <...>/episodes/<T?_####>/<file> of this owner's tasks -> [(owner/episodes/<id>/<file>, id, size)]."""
    out, where = {}, {}
    for p, fid, size in files:
        parts = p.split("/")
        if "episodes" not in parts[:-2]: continue
        i = len(parts) - 1 - parts[::-1].index("episodes")
        if len(parts) < i + 3: continue
        eid = parts[i + 1]
        if not eid.startswith(OWNER_TASKS[owner]): continue
        base = "/".join(parts[:i])
        if where.setdefault(eid, base) != base: continue        # the same episode in two places: keep the first
        out[f"{owner}/episodes/{eid}/" + "/".join(parts[i + 2:])] = (fid, size)
    dup = len({b for b in where.values()})
    return [(k, *v) for k, v in sorted(out.items())], dup

def download(item):
    rel, fid, size = item; dst = os.path.join(STAGE, rel)
    if os.path.exists(dst) and (size is None or os.path.getsize(dst) == size): return os.path.getsize(dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with io.FileIO(dst + ".part", "wb") as fh:
        d, done = MediaIoBaseDownload(fh, svc().files().get_media(fileId=fid, **ALL), chunksize=64 << 20), False
        while not done: _, done = d.next_chunk(num_retries=5)
    os.replace(dst + ".part", dst); return os.path.getsize(dst)

def audit(owner):
    """Same check as the generator's audit: _COMPLETE.json present and every listed file has its recorded size."""
    root = os.path.join(STAGE, owner, "episodes"); ok, bad = [], []
    for eid in sorted(os.listdir(root)):
        m = os.path.join(root, eid, "_COMPLETE.json")
        if not os.path.exists(m): bad.append((eid, "no _COMPLETE.json")); continue
        recs = json.load(open(m))["sha256"]
        miss = [n for n, r in recs.items() if not os.path.exists(os.path.join(root, eid, n))
                or os.path.getsize(os.path.join(root, eid, n)) != r["bytes"]]
        (bad.append((eid, "bad/missing " + ", ".join(miss[:3]))) if miss else ok.append(eid))
    return ok, bad

def drive_folder(name):
    r = svc().files().list(q=f"name='{name}' and mimeType='{FOLDER}' and 'root' in parents and trashed=false",
                           fields="files(id)").execute(num_retries=5)["files"]
    return r[0]["id"] if r else svc().files().create(body={"name": name, "mimeType": FOLDER}, fields="id").execute()["id"]

def upload(path, parent):
    name, size = os.path.basename(path), os.path.getsize(path)
    old = svc().files().list(q=f"name='{name}' and '{parent}' in parents and trashed=false",
                             fields="files(id,size)").execute(num_retries=5)["files"]
    if old and int(old[0].get("size", -1)) == size: print(f"{name}: already uploaded"); return old[0]["id"]
    req = svc().files().create(body={"name": name, "parents": [parent]}, fields="id",
                               media_body=MediaFileUpload(path, chunksize=256 << 20, resumable=True))
    bar, resp = tqdm(total=size, unit="B", unit_scale=True, desc=f"upload {name}"), None
    while resp is None:
        status, resp = req.next_chunk(num_retries=5)
        if status: bar.update(status.resumable_progress - bar.n)
    bar.update(size - bar.n); bar.close(); return resp["id"]

def share(fid):
    if SHARE_PUBLIC:
        try: svc().permissions().create(fileId=fid, body={"type": "anyone", "role": "reader"}, **ALL).execute(num_retries=5)
        except Exception as e: print("could not share automatically (share it by hand):", str(e)[:200])
    return f"https://drive.google.com/file/d/{fid}/view"
print("Drive API ready")

Drive API ready


In [3]:
# ---- 1. list + download (parallel), 2. audit, 3. tar
os.makedirs(STAGE, exist_ok=True); os.makedirs(OUT, exist_ok=True); TARS = {}
for owner, fid in FOLDERS.items():
    t0 = time.time(); files = walk(fid); items, nbase = select(files, owner)
    eps = {r.split("/")[2] for r, _, _ in items}
    print(f"{owner}: {len(files)} files listed, {len(eps)} episodes ({len(items)} files) under {nbase} folder(s) "
          f"[{time.time() - t0:.0f}s]")
    assert items, f"no {owner}/episodes/<id>/ files found under folder {fid}"
    total = sum(s or 0 for _, _, s in items)
    with ThreadPoolExecutor(WORKERS) as pool, tqdm(total=total, unit="B", unit_scale=True, desc=f"download {owner}") as bar:
        for n in pool.map(download, items): bar.update(n)
    ok, bad = audit(owner)
    print(f"{owner}: {len(ok)} complete episodes, {len(bad)} incomplete", bad[:10])
    tar = os.path.join(OUT, f"{owner}.tar")
    if not os.path.exists(tar):
        t0 = time.time(); subprocess.run(["tar", "-cf", tar + ".part", "-C", STAGE, owner], check=True)
        os.replace(tar + ".part", tar); print(f"{tar}: {os.path.getsize(tar) / 1e9:.2f} GB [{time.time() - t0:.0f}s]")
    TARS[owner] = tar

Lee: 13606 files listed, 400 episodes (6800 files) under 1 folder(s) [7s]


download Lee:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Lee: 400 complete episodes, 0 incomplete []


/content/out/Lee.tar: 3.14 GB [15s]


Lazizbek: 6802 files listed, 400 episodes (6800 files) under 1 folder(s) [4s]


download Lazizbek:   0%|          | 0.00/3.18G [00:00<?, ?B/s]

Lazizbek: 400 complete episodes, 0 incomplete []
/content/out/Lazizbek.tar: 3.19 GB [15s]


In [4]:
# ---- 4. upload + share; also share the omni_v1 cache
parent = drive_folder(OUT_FOLDER_NAME); LINKS = {}
for owner, tar in TARS.items():
    LINKS[owner] = share(upload(tar, parent))
c = svc().files().list(q=f"name='{CACHE_NAME}' and 'me' in owners and trashed=false",
                       fields="files(id,size,modifiedTime)", orderBy="modifiedTime desc").execute(num_retries=5)["files"]
if c: LINKS["cache"] = share(c[0]["id"])
else: print(f"{CACHE_NAME} not found in My Drive")
print(json.dumps(LINKS, indent=1))
print("\nOn the server:\n  bash jobs/download.sh " + " ".join(f"'{LINKS.get(k, '')}'" for k in ("Lee", "Lazizbek", "cache")))

upload Lee.tar:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

upload Lazizbek.tar:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

{
 "Lee": "https://drive.google.com/file/d/1zsm-UuLovJfIR4_LxqJ7f0Q9VcLFNZZX/view",
 "Lazizbek": "https://drive.google.com/file/d/1DJtEeGLXfFq6eiiNVe_dj_ji4b207h1c/view",
 "cache": "https://drive.google.com/file/d/1ya7KgHxGVM31AiK6k2n12qzC90uQsg77/view"
}

On the server:
  bash jobs/download.sh 'https://drive.google.com/file/d/1zsm-UuLovJfIR4_LxqJ7f0Q9VcLFNZZX/view' 'https://drive.google.com/file/d/1DJtEeGLXfFq6eiiNVe_dj_ji4b207h1c/view' 'https://drive.google.com/file/d/1ya7KgHxGVM31AiK6k2n12qzC90uQsg77/view'
